<a href="https://colab.research.google.com/github/mhtjsh/ViT-ImageSegmentation-Training/blob/Primary/q2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Housekeeping (to be executed once per session)

### CLIPSeg and SAM2 Installation

In [ ]:
!nvidia-smi || true

!pip install -q --upgrade pip
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

!pip install -q transformers Pillow matplotlib opencv-python

!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -q -e .

### Helper, Imports, Mask Upsampling and Model Loading

In [ ]:
# Core imports
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import cv2
import torch

# CLIPSeg imports (Hugging Face transformers)
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation

# SAM2 predictor (installed from the repo)
# This import will be valid after you `pip install -e sam2`.
from sam2.sam2_image_predictor import SAM2ImagePredictor

# Small helpers: load image, show overlay
def load_image(path, as_rgb=True):
    img = Image.open(path).convert("RGB") if as_rgb else Image.open(path)
    return img

def show_image_with_mask(pil_image, mask, alpha=0.45, title=None):
    img = np.array(pil_image).astype(np.uint8)
    H, W = mask.shape[:2]
    assert img.shape[0] == H and img.shape[1] == W, "mask and image size mismatch"
    overlay = img.copy()
    color = np.array([255, 0, 128], dtype=np.uint8)
    if mask.dtype == bool:
        overlay[mask] = (overlay[mask] * 0.35 + color * 0.65).astype(np.uint8)
    else:
        alpha_map = (mask[..., None] * alpha).astype(np.float32)
        overlay = (img * (1 - alpha_map) + color * alpha_map).astype(np.uint8)
    plt.figure(figsize=(10,6))
    plt.imshow(overlay)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

def upsample_mask_to_image(mask_small, image_size):
    mask_resized = cv2.resize((mask_small*255.0).astype(np.uint8),
                              (image_size[0], image_size[1]),
                              interpolation=cv2.INTER_LINEAR)
    mask_resized = (mask_resized.astype(np.float32) / 255.0)
    return mask_resized

# Helper functions for sampling points
def sample_points_in_mask(mask_bin, n_points=1):
    """Samples points uniformly from within the mask."""
    ys, xs = np.where(mask_bin)
    if len(ys) == 0:
        return np.empty((0, 2), dtype=np.float32)
    indices = np.random.choice(len(ys), size=min(n_points, len(ys)), replace=False)
    points = np.stack([xs[indices], ys[indices]], axis=1).astype(np.float32)
    return points

def sample_points_outside_mask(mask_bin, n_bg=1):
    """Samples points uniformly from outside the mask."""
    H, W = mask_bin.shape
    mask_inv = ~mask_bin
    ys, xs = np.where(mask_inv)
    if len(ys) == 0:
        return np.empty((0, 2), dtype=np.float32)
    indices = np.random.choice(len(ys), size=min(n_bg, len(ys)), replace=False)
    points = np.stack([xs[indices], ys[indices]], axis=1).astype(np.float32)
    return points

def keep_largest_component(mask_bin):
    """Keeps only the largest connected component in a binary mask."""
    num_labels, labels_im = cv2.connectedComponents(mask_bin.astype(np.uint8))
    if num_labels < 2:
        return mask_bin  # No components or only one (background)
    areas = [(labels_im == i).sum() for i in range(1, num_labels)]
    largest_component_label = np.argmax(areas) + 1
    return (labels_im == largest_component_label)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ---- CLIPSeg (Hugging Face) ----
clipseg_model_id = "CIDAS/clipseg-rd64-refined"
processor = CLIPSegProcessor.from_pretrained(clipseg_model_id)
clipseg_model = CLIPSegForImageSegmentation.from_pretrained(clipseg_model_id).to(device)
clipseg_model.eval()

# ---- SAM 2 predictor ----
sam_checkpoint = "facebook/sam2-hiera-large"
predictor = SAM2ImagePredictor.from_pretrained(sam_checkpoint, device=device)


### Dataset Installation
  - For this experiment, I am using the ``COCO 2017``  dataset which is widely used validation dataset for image segmentation, named as ``val2017`` with instance segmentation annotation.

In [ ]:
# === Dataset download & setup ===
# We’ll download COCO 2017 images (val) + instance segmentation annotations

!mkdir -p dataset/coco
%cd dataset/coco

# Download validation images (about 5K images)
!wget -q http://images.cocodataset.org/zips/val2017.zip
!unzip -q val2017.zip
!rm val2017.zip

# Download instance annotations
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q annotations_trainval2017.zip
!rm annotations_trainval2017.zip

%cd ../../

!echo "COCO dataset setup done."

In [ ]:
import os
import json
from PIL import Image
import numpy as np
from pycocotools import mask as mask_utils # Import pycocotools here

# Paths (adjust if your directory differs)
COCO_ROOT = "dataset/coco"
IMG_DIR = os.path.join(COCO_ROOT, "val2017")
ANN_FILE = os.path.join(COCO_ROOT, "annotations", "instances_val2017.json")

# Load COCO annotations JSON
with open(ANN_FILE, "r") as f:
    coco_ann = json.load(f)

# Build an index: image_id → list of annotation dicts
imgid2anns = {}
for ann in coco_ann["annotations"]:
    imgid = ann["image_id"]
    imgid2anns.setdefault(imgid, []).append(ann)

# Build image metadata map: image_id → file_name, width, height
imgid2meta = {img["id"]: img for img in coco_ann["images"]}

def load_coco_image_and_gt(img_id):
    """
    Returns (PIL image, list of binary masks, list of category names, image metadata)
    """
    meta = imgid2meta[img_id]
    fname = meta["file_name"]
    path = os.path.join(IMG_DIR, fname)
    img = Image.open(path).convert("RGB")

    anns = imgid2anns.get(img_id, [])
    masks = []
    cat_names = []
    for ann in anns:
        # ann["segmentation"] can be polygon or RLE; use pycocotools to convert
        h, w = meta["height"], meta["width"]
        segm = ann["segmentation"]
        if isinstance(segm, list):
            # polygon -> RLE
            rle = mask_utils.frPyObjects(segm, h, w)
            m = mask_utils.decode(rle)
            # If multiple RLE, sum (they don’t overlap by COCO design)
            if isinstance(m, list):
                m = np.sum(m, axis=2)
        else:
            # already RLE
            m = mask_utils.decode(segm)
        # Convert >0 to boolean and ensure 2D shape
        mask_bin = (m > 0)
        if mask_bin.ndim == 3 and mask_bin.shape[2] == 1:
             mask_bin = mask_bin.squeeze(axis=2)
        masks.append(mask_bin)
        cat_id = ann["category_id"]
        # Find category name safely
        cat_name = "unknown"
        for category in coco_ann["categories"]:
            if category["id"] == cat_id:
                cat_name = category["name"]
                break
        cat_names.append(cat_name)

    return img, masks, cat_names, meta

# Example: list first 5 image ids
example_ids = list(imgid2meta.keys())[:5]
print("Example COCO image IDs:", example_ids)

In [ ]:
!pip install -q pycocotools

## Usage from this cell onwards

### CLIPSeg Image Seg Check on random COCO 2017 dataset

In [ ]:
import random
img_id = random.choice(example_ids)
pil_img, gt_masks, cat_names, meta = load_coco_image_and_gt(img_id)
print("Image ID:", img_id, "Categories present:", set(cat_names))

text_prompt = cat_names[0]
print("Using prompt:", text_prompt)

inputs = processor(text=[text_prompt], images=pil_img, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = clipseg_model(**inputs)
logits = outputs.logits.cpu().squeeze(0).squeeze(0)
probs = torch.sigmoid(logits).numpy()

W, H = pil_img.size
mask_up = upsample_mask_to_image(probs, (W, H))
show_image_with_mask(pil_img, mask_up, alpha=0.5, title=f"CLIPSeg coarse mask for '{text_prompt}'")


Point extraction from the CLIPSeg

In [ ]:
# Binarize
thresh = 0.35
mask_bin = mask_up > thresh

# Keep largest component
num_labels, labels_im = cv2.connectedComponents(mask_bin.astype(np.uint8))
areas = [(labels_im == i).sum() for i in range(num_labels)]
if num_labels > 1:
    largest_component = np.argmax(areas[1:]) + 1
    mask_bin = (labels_im == largest_component)

ys, xs = np.where(mask_bin)
if len(xs) == 0:
    print("Empty mask at threshold", thresh)
    bbox = None
else:
    xmin, xmax = int(xs.min()), int(xs.max())
    ymin, ymax = int(ys.min()), int(ys.max())
    bbox = [xmin, ymin, xmax, ymax]
    print("Derived bbox:", bbox)

# Sample points
fg_points = sample_points_in_mask(mask_bin, n_points=30)
bg_points = sample_points_outside_mask(mask_bin, n_bg=10)
print("FG pts:", fg_points.shape, "BG pts:", bg_points.shape)


### SAM 2 refinement

In [ ]:
# Use the large SAM2 checkpoint you installed
# Your `predictor` is already loaded with that large checkpoint in an earlier cell

img_np = np.array(pil_img)
predictor.set_image(img_np)

points = fg_points
labels = np.ones(len(points), dtype=int)
if len(bg_points) > 0:
    points = np.vstack([points, bg_points])
    labels = np.concatenate([labels, np.zeros(len(bg_points), dtype=int)])


with torch.inference_mode():
    masks_out, scores_out, _ = predictor.predict(
        point_coords=points,
        point_labels=labels,
        multimask_output=True
    )

# Choose best mask
if isinstance(scores_out, (list, np.ndarray)) and len(scores_out) > 0:
    best_idx = int(np.argmax(scores_out))
else:
    best_idx = 0

if isinstance(masks_out, np.ndarray):
    sam_mask = masks_out[best_idx].astype(bool)
else:
    sam_mask = np.array(masks_out[best_idx]).astype(bool)


show_image_with_mask(pil_img, sam_mask, alpha=0.5, title=f"SAM2 refined mask for '{text_prompt}' (Large)")

In [ ]:
# Clean SAM mask
sam_mask_clean = keep_largest_component(sam_mask)
show_image_with_mask(pil_img, sam_mask_clean, alpha=0.6, title="Final cleaned SAM mask")

# Optionally compute IoU / evaluation with ground truth masks
def compute_iou(bin1, bin2):
    inter = np.logical_and(bin1, bin2).sum()
    union = np.logical_or(bin1, bin2).sum()
    if union == 0:
        return 0.0
    return inter / union

# Among ground truth masks, pick the one with max IoU with our predicted
ious = [compute_iou(sam_mask_clean, gt) for gt in gt_masks]
best_gt_idx = int(np.argmax(ious)) if len(ious) > 0 else None
best_iou = ious[best_gt_idx] if best_gt_idx is not None else 0.0
print("Best IoU with GT:", best_iou, "GT category:", cat_names[best_gt_idx] if best_gt_idx is not None else "-")

